# Leakage prevention

The previous version of the dataset created the target_7d for the last 7 rows of the train dataset, which means that there are leakage 


In [31]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [32]:
TRAIN_PATH = Path(r"outputs/data/train_df.csv")
VAL_PATH = Path(r"outputs/data/val_df.csv")
TEST_PATH = Path(r"outputs/data/test_df.csv")

OUTPUT_DIR = Path("outputs/lstm_ready_data")
OUTPUT_DIR.mkdir(exist_ok=True)

LOOKBACK = 30          
HORIZON = 7            
SCALE_METHOD = "zscore"   

TARGET_COL = "target_7d"
ID_COLS = ["crypto", "date"]

CONTINUOUS_FEATURES = [
    "close", "log_return", "open", "high", "low", "volume",
    "vol_30", "sma", "ema", "momentum", "cci"
]

BINARY_FEATURES = ["regime_label"]
FEATURE_COLS = CONTINUOUS_FEATURES + BINARY_FEATURES

In [33]:
train_df = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
val_df = pd.read_csv(VAL_PATH, parse_dates=["date"])
test_df = pd.read_csv(TEST_PATH, parse_dates=["date"])

def sort_panel(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(["crypto", "date"]).reset_index(drop=True)

train_df = sort_panel(train_df)
val_df = sort_panel(val_df)
test_df = sort_panel(test_df)

print("Loaded shapes:")
print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

Loaded shapes:
train: (11448, 18)
val:   (2928, 18)
test:  (2864, 18)



## Verification helpers

Check that every asset has same number of rows, same ordered timestamp, dates are ordered properly, no missing dates, train test validation not overlap

In [34]:
def verify_alignment(df: pd.DataFrame, split_name: str) -> dict:
    df = sort_panel(df)

    counts = df.groupby("crypto").size()
    equal_lengths = counts.nunique() == 1

    ordered_dates = {
        asset: tuple(sub["date"])
        for asset, sub in df.groupby("crypto", sort=True)
    }
    first_dates = next(iter(ordered_dates.values()))
    same_ordered_timestamps = all(dates == first_dates for dates in ordered_dates.values())

    strictly_increasing_dates = all(
        sub["date"].is_monotonic_increasing
        for _, sub in df.groupby("crypto", sort=True)
    )

    coverage = (
        df.assign(value=1)
          .pivot(index="date", columns="crypto", values="value")
    )
    no_missing_asset_date_pairs = not coverage.isna().any().any()

    summary = {
        "split": split_name,
        "n_assets": int(df["crypto"].nunique()),
        "n_dates": int(df["date"].nunique()),
        "rows_per_asset_min": int(counts.min()),
        "rows_per_asset_max": int(counts.max()),
        "equal_lengths": bool(equal_lengths),
        "same_ordered_timestamps": bool(same_ordered_timestamps),
        "strictly_increasing_dates": bool(strictly_increasing_dates),
        "no_missing_asset_date_pairs": bool(no_missing_asset_date_pairs),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
    }
    return summary


def check_split_boundaries(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    train_max = train_df["date"].max()
    val_min = val_df["date"].min()
    val_max = val_df["date"].max()
    test_min = test_df["date"].min()

    assert train_max < val_min, "Train and validation dates overlap."
    assert val_max < test_min, "Validation and test dates overlap."


def trim_split_for_horizon(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    trimmed = []
    for asset, sub in df.groupby("crypto", sort=True):
        sub = sub.sort_values("date").copy()
        if len(sub) <= horizon:
            raise ValueError(f"{asset} has <= horizon rows and cannot be trimmed safely.")
        trimmed.append(sub.iloc[:-horizon].copy())
    return pd.concat(trimmed, ignore_index=True)

Boundary Check and Trim:

In [35]:
check_split_boundaries(train_df, val_df, test_df)

train_safe = trim_split_for_horizon(train_df, HORIZON)
val_safe = trim_split_for_horizon(val_df, HORIZON)
test_safe = test_df.copy()   

verification_rows = [
    verify_alignment(train_safe, "train_safe"),
    verify_alignment(val_safe, "val_safe"),
    verify_alignment(test_safe, "test_safe"),
]
verification_df = pd.DataFrame(verification_rows)
verification_df

,split,n_assets,n_dates,rows_per_asset_min,rows_per_asset_max,equal_lengths,same_ordered_timestamps,strictly_increasing_dates,no_missing_asset_date_pairs,date_min,date_max
0,train_safe,8,1424,1424,1424,True,True,True,True,2020-01-31,2023-12-24
1,val_safe,8,359,359,359,True,True,True,True,2024-01-01,2024-12-24
2,test_safe,8,358,358,358,True,True,True,True,2025-01-01,2025-12-24


## Scaling

scaler (now using zscaler through values obtained from train set but applied to all)


In [36]:
def get_scaler(method: str):
    if method == "zscore":
        return StandardScaler()
    if method == "minmax":
        return MinMaxScaler()
    raise ValueError("SCALE_METHOD must be either 'zscore' or 'minmax'.")


def scale_splits(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    continuous_features: list[str],
    method: str
):
    scaler = get_scaler(method)

    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    test_scaled = test_df.copy()

    train_scaled[continuous_features] = scaler.fit_transform(train_df[continuous_features])
    val_scaled[continuous_features] = scaler.transform(val_df[continuous_features])
    test_scaled[continuous_features] = scaler.transform(test_df[continuous_features])

    return train_scaled, val_scaled, test_scaled, scaler


train_scaled, val_scaled, test_scaled, scaler = scale_splits(
    train_safe, val_safe, test_safe,
    continuous_features=CONTINUOUS_FEATURES,
    method=SCALE_METHOD
)

print("Scaler fitted on train only.")
print("Scaled train shape:", train_scaled.shape)
print("Scaled val shape:  ", val_scaled.shape)
print("Scaled test shape: ", test_scaled.shape)

Scaler fitted on train only.
Scaled train shape: (11392, 18)
Scaled val shape:   (2872, 18)
Scaled test shape:  (2864, 18)



## LSTM sliding windows

Each sample uses the lookback days, ending at date `t`, and predicts the already-prepared target on date `t`.
So with `LOOKBACK = 30` and `target_7d`:
- **input sequence** = days `t-29` to `t`
- **target** = cumulative return from `t+1` to `t+7`

### Context across split boundaries
For validation and test, the first sequences still need past days from the previous split.  
That is safe, so this notebook uses:
- validation history = train
- test history = train + validation

The target row itself still belongs only to the current split.


In [37]:
def build_lstm_sequences(
    current_df: pd.DataFrame,
    history_df: pd.DataFrame | None,
    feature_cols: list[str],
    target_col: str,
    lookback: int
):
    current_df = sort_panel(current_df)
    history_df = sort_panel(history_df) if history_df is not None else None

    X_list = []
    y_list = []
    meta_rows = []

    for asset in sorted(current_df["crypto"].unique()):
        current_asset = current_df[current_df["crypto"] == asset].copy()

        if history_df is not None:
            history_asset = history_df[history_df["crypto"] == asset].copy()
            combined = pd.concat([history_asset, current_asset], ignore_index=True)
            combined = (
                combined.drop_duplicates(subset=["date"], keep="last")
                        .sort_values("date")
                        .reset_index(drop=True)
            )
        else:
            combined = current_asset.copy()

        current_dates = set(current_asset["date"])

        values = combined[feature_cols].to_numpy(dtype=np.float32)
        targets = combined[target_col].to_numpy(dtype=np.float32)
        dates = combined["date"].to_numpy()
        cryptos = combined["crypto"].to_numpy()

        for end_idx in range(lookback - 1, len(combined)):
            # keep only targets whose forecast origin belongs to the current split
            if dates[end_idx] not in current_dates:
                continue

            seq = values[end_idx - lookback + 1 : end_idx + 1]
            if seq.shape[0] != lookback:
                continue

            X_list.append(seq)
            y_list.append(targets[end_idx])
            meta_rows.append((cryptos[end_idx], pd.Timestamp(dates[end_idx])))

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_rows, columns=["crypto", "date"])

    return X, y, meta


train_history = None
val_history = train_scaled.copy()
test_history = pd.concat([train_scaled, val_scaled], ignore_index=True)

X_train, y_train, meta_train = build_lstm_sequences(
    current_df=train_scaled,
    history_df=train_history,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    lookback=LOOKBACK,
)

X_val, y_val, meta_val = build_lstm_sequences(
    current_df=val_scaled,
    history_df=val_history,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    lookback=LOOKBACK,
)

X_test, y_test, meta_test = build_lstm_sequences(
    current_df=test_scaled,
    history_df=test_history,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    lookback=LOOKBACK,
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

X_train: (11160, 30, 12) y_train: (11160,)
X_val:   (2872, 30, 12) y_val:   (2872,)
X_test:  (2864, 30, 12) y_test:  (2864,)


In [38]:
assert X_train.shape[1] == LOOKBACK
assert X_val.shape[1] == LOOKBACK
assert X_test.shape[1] == LOOKBACK

assert X_train.shape[2] == len(FEATURE_COLS)
assert X_val.shape[2] == len(FEATURE_COLS)
assert X_test.shape[2] == len(FEATURE_COLS)

display(verification_df)

,split,n_assets,n_dates,rows_per_asset_min,rows_per_asset_max,equal_lengths,same_ordered_timestamps,strictly_increasing_dates,no_missing_asset_date_pairs,date_min,date_max
0,train_safe,8,1424,1424,1424,True,True,True,True,2020-01-31,2023-12-24
1,val_safe,8,359,359,359,True,True,True,True,2024-01-01,2024-12-24
2,test_safe,8,358,358,358,True,True,True,True,2025-01-01,2025-12-24


## Save processed outputs


In [39]:
np.save(OUTPUT_DIR / "X_train.npy", X_train)
np.save(OUTPUT_DIR / "y_train.npy", y_train)
np.save(OUTPUT_DIR / "X_val.npy", X_val)
np.save(OUTPUT_DIR / "y_val.npy", y_val)
np.save(OUTPUT_DIR / "X_test.npy", X_test)
np.save(OUTPUT_DIR / "y_test.npy", y_test)

## Per-Asset Data Processing for LSTM
You can process and save LSTM-ready data for each asset separately, enabling you to train a different LSTM model for each asset. The following code demonstrates how to loop through each asset, apply the same preprocessing pipeline, and save the results for each asset individually.

In [40]:
assets = train_df['crypto'].unique()

for asset in assets:
    print(f'Processing asset: {asset}')
    
    # filter dataset for asset
    train_asset = train_df[train_df['crypto'] == asset].copy()
    val_asset = val_df[val_df['crypto'] == asset].copy()
    test_asset = test_df[test_df['crypto'] == asset].copy()

    # trim for horizon
    train_safe = trim_split_for_horizon(train_asset, HORIZON)
    val_safe = trim_split_for_horizon(val_asset, HORIZON)
    test_safe = test_asset.copy()

    # scale
    train_scaled, val_scaled, test_scaled, scaler = scale_splits(
        train_safe, val_safe, test_safe,
        continuous_features=CONTINUOUS_FEATURES,
        method=SCALE_METHOD
    )

    # build LSTM sequences
    train_history = None
    val_history = train_scaled.copy()
    test_history = pd.concat([train_scaled, val_scaled], ignore_index=True)

    X_train, y_train, meta_train = build_lstm_sequences(
        current_df=train_scaled,
        history_df=train_history,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=LOOKBACK,
    )
    X_val, y_val, meta_val = build_lstm_sequences(
        current_df=val_scaled,
        history_df=val_history,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=LOOKBACK,
    )
    X_test, y_test, meta_test = build_lstm_sequences(
        current_df=test_scaled,
        history_df=test_history,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=LOOKBACK,
    )

    # save processed outputs
    np.save(OUTPUT_DIR / f"X_train_{asset}.npy", X_train)
    np.save(OUTPUT_DIR / f"y_train_{asset}.npy", y_train)
    np.save(OUTPUT_DIR / f"meta_train_{asset}.npy", meta_train)
    np.save(OUTPUT_DIR / f"X_val_{asset}.npy", X_val)
    np.save(OUTPUT_DIR / f"y_val_{asset}.npy", y_val)
    np.save(OUTPUT_DIR / f"meta_val_{asset}.npy", meta_val)
    np.save(OUTPUT_DIR / f"X_test_{asset}.npy", X_test)
    np.save(OUTPUT_DIR / f"y_test_{asset}.npy", y_test)
    np.save(OUTPUT_DIR / f"meta_test_{asset}.npy", meta_test)
    print(f'Saved LSTM arrays for {asset}')

    # check shapes
    print(f'Shapes for {asset}:')
    print(f'X_train: {X_train.shape}, y_train: {y_train.shape}, meta_train: {meta_train.shape}')
    print(f'X_val: {X_val.shape}, y_val: {y_val.shape}, meta_val: {meta_val.shape}')
    print(f'X_test: {X_test.shape}, y_test: {y_test.shape}, meta_test: {meta_test.shape}')
    print("\n")

Processing asset: ADAUSDT
Saved LSTM arrays for ADAUSDT
Shapes for ADAUSDT:
X_train: (1395, 30, 12), y_train: (1395,), meta_train: (1395, 2)
X_val: (359, 30, 12), y_val: (359,), meta_val: (359, 2)
X_test: (358, 30, 12), y_test: (358,), meta_test: (358, 2)


Processing asset: BCHUSDT
Saved LSTM arrays for BCHUSDT
Shapes for BCHUSDT:
X_train: (1395, 30, 12), y_train: (1395,), meta_train: (1395, 2)
X_val: (359, 30, 12), y_val: (359,), meta_val: (359, 2)
X_test: (358, 30, 12), y_test: (358,), meta_test: (358, 2)


Processing asset: BNBUSDT
Saved LSTM arrays for BNBUSDT
Shapes for BNBUSDT:
X_train: (1395, 30, 12), y_train: (1395,), meta_train: (1395, 2)
X_val: (359, 30, 12), y_val: (359,), meta_val: (359, 2)
X_test: (358, 30, 12), y_test: (358,), meta_test: (358, 2)


Processing asset: BTCUSDT
Saved LSTM arrays for BTCUSDT
Shapes for BTCUSDT:
X_train: (1395, 30, 12), y_train: (1395,), meta_train: (1395, 2)
X_val: (359, 30, 12), y_val: (359,), meta_val: (359, 2)
X_test: (358, 30, 12), y_test